# Process A Longstrip - Data Part E - Longstrip Post-Metrics

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [ ]:
# Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;
codename = '20250505';

#folder_oct
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = r'D:\SGProjects\NAATOS\OCTlocal'

#study_name = 'GHL_pyapp_20250505T1514'; # oct study folder
#study_name = 'GHL_pyapp_20250505T1523'; # oct study folder
study_name = 'GHL_pyapp_20250505T1531'; # oct study folder
#study_name = 'GHL_pyapp_20250505T1542'; # oct study folder

folder_figure_temp = r'D:\\TEMP\\OCTtmp\\';

In [ ]:
folder_octexport_root = Path(folder_octexport_root);

# Load the OCT Study

In [3]:
octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(study_name,folder_octexport_root);

STUDY: GHL_pyapp_20250505T1531
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 20,
    'study_num_vtk_files': 0}


In [4]:
# Prepare a temporary location to store images as we go through each slice
folder_figure_temp = Path(f'C:\\TEMP\\OCTtmp\\{octstudy.name}');
folder_figure_temp.mkdir(parents=True,exist_ok=True);

# 1. From stored dataframes load summary data we care about

In [5]:
print(octstudy.name)

octstudy.load_previously_saved_merged_volume();

data_extracted = octstudy.load_data_extracted();
dfsteps = octstudy.load_data_extracted_along_strip();


GHL_pyapp_20250505T1531
Loading GHL_pyapp_20250505T1531_STACKED_RESCALED_30to60_uint8.vtk
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE


In [6]:
dfsteps

,pixel_depth_strip_top_sum_threshold,pixel_depth_strip_top,pixel_depth_strip_bot,pixel_depth_wax_center,px_wax_transverse_edges,wax_top_seed_candidate_px,pksA,pksB,pksB2,wax_roi_valve_bottom,testseg_otsu_regions,testseg_otsu_thresholds,final_wax_bot_subset_px,seg_thresh_sauvola_cutoff,area,area_filled,area_convex,num_paths
slice,,,,,,,,,,,,,,,,,,
5,35.16,208,403,325,"(91.55555555555556, 120.11111111111111)","(380, 105)","([13, 76, 129, 169], {'prominences': [1674.0, ...","([20, 56, 86, 131, 179], {'prominences': [1798...","([131, 185], {'peak_heights': [186.35306833524...",179,"[[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3,...","[84, 117, 149, 184]",216,"[[False, False, False, False, False, False, Fa...",256.0,256.0,338.0,0
10,51.60,203,264,238,"(62.327586206896555, 117.39795918367346)","(218, 89)","([13, 46, 89, 119, 165], {'prominences': [7594...","([17, 180, 216], {'prominences': [5185.0, 100....","([22, 216], {'peak_heights': [2380.98194328483...",216,"[[1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3,...","[58, 93, 129, 176]",216,"[[False, False, False, False, False, False, Fa...",6413.0,7195.0,8959.0,0
15,65.48,204,258,240,"(64.92857142857143, 116.4140625)","(216, 89)","([9, 53, 93], {'prominences': [8827.0, 331.0, ...","([9, 103, 152, 216], {'prominences': [2720.0, ...","([10, 216], {'peak_heights': [4746.18011097195...",216,"[[3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,...","[61, 97, 139, 199]",216,"[[False, False, False, False, False, False, Fa...",6885.0,7689.0,9095.0,0
20,64.76,203,258,243,"(65.9364406779661, 116.83653846153847)","(212, 90)","([9, 54, 93], {'prominences': [9768.0, 179.0, ...","([10, 97, 152, 216], {'prominences': [4575.0, ...","([10, 102, 216], {'peak_heights': [5188.068852...",216,"[[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,...","[60, 92, 129, 189]",217,"[[False, False, False, False, False, False, Fa...",7462.0,8641.0,9733.0,0
25,60.80,203,258,239,"(65.58712121212122, 120.67424242424242)","(220, 92)","([9, 43, 95], {'prominences': [9493.0, 484.0, ...","([10, 95, 217, 401], {'prominences': [5551.0, ...","([10, 95, 217], {'peak_heights': [5413.6219762...",95,"[[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,...","[56, 89, 131, 191]",220,"[[False, False, False, False, False, False, Fa...",8301.0,9406.0,10872.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8230,57.40,303,363,335,"(61.71470588235294, 130.5900900900901)","(318, 95)","([15, 64, 147], {'prominences': [11703.0, 1864...","([17, 152, 262], {'prominences': [11349.0, 522...","([17, 152, 263], {'peak_heights': [4441.768558...",152,"[[0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 4, 4,...","[58, 100, 149, 201]",152,"[[False, False, False, False, False, False, Fa...",6458.0,6557.0,8448.0,0
8235,58.16,300,360,334,"(62.747899159663866, 131.0229007633588)","(318, 96)","([16, 77, 112, 150], {'prominences': [10862.0,...","([20, 152, 264], {'prominences': [10906.0, 475...","([21, 155, 264], {'peak_heights': [4238.057678...",155,"[[1, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3,...","[60, 103, 150, 202]",155,"[[False, False, False, False, False, False, Fa...",6376.0,6401.0,8727.0,0
8240,59.48,298,355,334,"(62.663636363636364, 130.65032679738562)","(316, 96)","([16, 53, 86, 153], {'prominences': [10861.0, ...","([22, 156, 265], {'prominences': [9613.0, 4177...","([22, 158, 266], {'peak_heights': [3834.552224...",158,"[[2, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,...","[64, 105, 152, 203]",158,"[[False, False, False, False, False, False, Fa...",6485.0,6578.0,8750.0,0


# Prepare the source image for convenience

In [7]:
# Test getting SITK image
# Get ITK image without requiring new memory
import itk
import vtk

# function to go from itk image to simpleitk image
def itkToSimpleITK(itk_image):
    new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
    new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
    new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
    new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
    return new_sitk_image;

# create simpleitk 3dimage, directly from the previously-loaded vtk 3dimage
simgmerged = itkToSimpleITK( itk.image_from_vtk_image(octstudy.vdvol.dataset) );



# itk_image
#print(itk_image)
slicer_1 = slice( data_extracted['auto_bounds']['topdown_edge_left_px3']  ,  data_extracted['auto_bounds']['topdown_edge_right_px3']  )
#print(' slicer_dim_0',slicer_0)
print(' slicer_dim_1',slicer_1)
#simgmerged.GetSize()

 slicer_dim_1 slice(249, 8504, None)


# 2. Calculate paths through segmentation

In [8]:
import matplotlib.patches
def mkfig(slice_position_along_strip_px):
    data={};
    #sliceinfo = dfsteps.iloc[412];
    sliceinfo = dfsteps.loc[slice_position_along_strip_px];
    sliceseg = sliceinfo['seg_thresh_sauvola_cutoff'];
    print('Shape',sliceseg.shape)


    fig = plt.figure(figsize=(14,10));
    gs = fig.add_gridspec(4,2);
    fig.suptitle('{:s} --> range={:s}dB to uint8\ncheck connectivity from bottom to top\nslice_position_along_strip={:.0f}px'.format(octstudy.name,str((oct_scalar_min,oct_scalar_max)),sliceinfo.name))


    # will cut off / not consider connections too close to the strip surface
    #cutoff_depth = (sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top']);
    cutoff_depth = (sliceinfo['wax_top_seed_candidate_px'][0]-sliceinfo['pixel_depth_strip_top']) + (sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])//2
    # don't consider connectons near the "bottom" of the segmentation
    cutoff_wax_bottom = sliceseg.shape[1]-int((sliceseg.shape[1]-cutoff_depth)*0.175);

    ax = fig.add_subplot(gs[0,1]);
    ax.imshow(sliceseg,aspect='auto',cmap='gray')
    ax.axvline(cutoff_depth-0.5,color='cyan')
    ax.set_title('#1. Previously-computed Segmentation of Wax Region',color='magenta',y=0.95,verticalalignment='top')

    bot_side = sliceseg.shape[0];
    top_side = 0;
    ntracebacks = 0;

    img = ~sliceseg;
    img[:,0:cutoff_depth] = False
    img[:,cutoff_wax_bottom:] = False

    costarray = (img.astype(int))-1;
    costarray[costarray==0] = 1;
    costarray[:,0:cutoff_depth] = -1
    costarray[:,cutoff_wax_bottom:] = -1;

    ax = fig.add_subplot(gs[1,1]);
    ax.imshow(costarray,aspect='auto',cmap='gray')
    #ax.axvline((sliceinfo['pixel_depth_strip_bot']-sliceinfo['pixel_depth_strip_top'])-0.5)
    ax.axvline(cutoff_depth-0.5,color='cyan')
    ax.set_title("#2. Cost matrix (from #1) but left of\nsliceinfo['wax_top_seed_candidate_px'][0]is set to infinite",color='magenta',y=0.95,verticalalignment='top')



    # graph, find path from shortest to longest
    top_source_points_col = np.where(costarray[top_side,:]==1)[0]
    top_source_points_row = np.ones_like(top_source_points_col)*top_side;
    top_source_points = np.stack([top_source_points_row,top_source_points_col],axis=1)
    #top_source_points = (top_source_points_row,top_source_points_col);


    bot_source_points_col = np.where(costarray[bot_side-1,:]==1)[0]
    bot_source_points_row = np.ones_like(bot_source_points_col)*(bot_side-1);
    bot_source_points = np.stack([bot_source_points_row,bot_source_points_col],axis=1)

    goodtraceback = False;

    if(top_source_points.shape[0]!=0 and bot_source_points.shape[0]!=0):
            


        #ret = skimage.graph.route_through_array(~sliceseg,start=(bot_source_points_row[0],bot_source_points_col[0]),end=(top_source_points_row[0],top_source_points_col[0]))
        #ret = skimage.graph.route_through_array(~sliceseg,start=(bot_source_points_row[0],bot_source_points_col[0]),end=(top_source_points_row[0],top_source_points_col[0]))


        # Sk Image Graph MCP Module
        #m = skimage.graph.MCP( costarray , fully_connected=True );
        m = skimage.graph.MCP_Geometric( costarray ,fully_connected=True);
        #ret = m.find_costs(starts=[bot_source_points.tolist()[0]],ends=[top_source_points.tolist()[0]],find_all_ends=True);
        #ret = m.find_costs(starts=bot_source_points.tolist(),ends=[[tuple(x) for x in top_source_points.tolist()][0]],find_all_ends=False);
        ret = m.find_costs(starts=bot_source_points.tolist(),ends=top_source_points.tolist(),find_all_ends=True);
        #ret = m.find_costs(starts=top_source_points.tolist(),ends=bot_source_points.tolist(),find_all_ends=True);

        print('Min Top Col',np.min(ret[0][top_side,:]));
        print('Max Top Col',np.max(ret[0][top_side,:]));
        idxmin_end_col = np.argmin(ret[0][top_side,:]);
        idxmax_end_col = np.argmax(ret[0][top_side,:]);

        print('idxmin_end_col path',idxmin_end_col);
        print('idxmax_end_col path',idxmax_end_col);

        if(np.min(ret[0][top_side,:]) != np.inf):
            if False:
                traceback = m.traceback((top_side,idxmin_end_col))
                print(traceback)
                ax.plot( np.array(traceback)[:,1]-0 , np.array(traceback)[:,0]-0 , color='red' )
            if False:
                ntracebacks = np.where(~np.isinf( ret[0][bot_side-1,:] ))[0].shape[0];
                traceback = m.traceback((top_side,idxmin_end_col))
                print(traceback)
                ax.plot( np.array(traceback)[:,1]-0 , np.array(traceback)[:,0]-0 , color='red' )
                ax.text(
                    x=0.05,  # x-coordinate (0 is left, 1 is right)
                    y=0.5,  # y-coordinate (0 is bottom, 1 is top)
                    s="Num Paths\nacross {:d}".format(ntracebacks),  # Text string
                    fontsize=15,
                    ha="left",  # Horizontal alignment
                    va="top",  # Vertical alignment
                    color='yellow',
                    transform=ax.transAxes # Specify that coordinates are axis-normalized
                )
            if False:
                ntracebacks = 0
                #for col in np.where(~np.isinf( ret[0][bot_side-1,:] ))[0]:
                for col in top_source_points_col:
                    try:
                        traceback = m.traceback((top_side,col))
                        ntracebacks+=1;
                        print(traceback)
                    except ValueError:
                        traceback = None
                    if(traceback is not None):
                        ax.plot( np.array(traceback)[:,1]-0 , np.array(traceback)[:,0]-0 , color='red' )
                ax.text(
                    x=0.05,  # x-coordinate (0 is left, 1 is right)
                    y=0.5,  # y-coordinate (0 is bottom, 1 is top)
                    s="Num Paths\nacross {:d}".format(ntracebacks),  # Text string
                    fontsize=15,
                    ha="left",  # Horizontal alignment
                    va="top",  # Vertical alignment
                    color='yellow',
                    transform=ax.transAxes # Specify that coordinates are axis-normalized
                )

                # for col in np.where(~np.isinf( ret[0][bot_side-1,:] ))[0]:
                #     try:
                #         traceback = m.traceback((top_side))
                #         tracebacks+=1;
                #     except ValueError:
                #         pass;
            if True:
                # setup networkx graph from pixels
                retg = skimage.graph.pixel_graph(img,sparse_type='array',connectivity=4)
                G = nx.from_scipy_sparse_array(retg[0])

                # compare raveled indices of top and bottom points, get graph node numbers
                raveled_graph_idx_node = retg[1][np.array(range(retg[0].shape[0]))];
                top_source_ravelidx = np.ravel_multi_index( (top_source_points[:,0],top_source_points[:,1]) , sliceseg.shape)
                print(top_source_ravelidx)
                bot_source_ravelidx = np.ravel_multi_index( (bot_source_points[:,0],bot_source_points[:,1]) , sliceseg.shape)
                print(bot_source_ravelidx)
                nodes_top = np.where(np.isin(raveled_graph_idx_node,top_source_ravelidx))[0]
                nodes_bot = np.where(np.isin(raveled_graph_idx_node,bot_source_ravelidx))[0]

                ntracebacks = 0
                # for col in top_source_points_col:
                #     try:
                #         traceback = m.traceback((top_side,col))
                #         ntracebacks+=1;
                #         print(traceback)
                #     except ValueError:
                #         traceback = None
                #     if(traceback is not None):
                #         ax.plot( np.array(traceback)[:,1]-0 , np.array(traceback)[:,0]-0 , color='red' )
                # for node in nodes_bot:
                #     try:
                #         r = nx.multi_source_dijkstra(G,sources=nodes_top.tolist(),target=node)
                #         print('Good path to',node);
                #         patharr = np.array(np.unravel_index(raveled_graph_idx_node[r[1]],shape=img.shape));
                #         ax.plot( patharr[1,:], patharr[0,:] , color='red' )
                #         ntracebacks+=1;
                #     except nx.NetworkXNoPath as e:
                #         print('No path to',node);
                # for node in nodes_top:
                #     try:
                #         r = nx.multi_source_dijkstra(G,sources=nodes_bot.tolist(),target=node)
                #         print('Good path to',node);
                #         patharr = np.array(np.unravel_index(raveled_graph_idx_node[r[1]],shape=img.shape));
                #         ax.plot( patharr[1,:], patharr[0,:] , color='red' )
                #         ntracebacks+=1;
                #     except nx.NetworkXNoPath as e:
                #         print('No path to',node);
                top_applicable_cols = np.where(~np.isinf( ret[0][top_side,:] ))[0];
                top_applicable_rows = np.ones_like(top_applicable_cols)*(top_side);
                top_nodes_applicable = np.where(np.isin(raveled_graph_idx_node,np.ravel_multi_index((top_applicable_rows,top_applicable_cols), sliceseg.shape)))[0];
                for nodetop in top_nodes_applicable:
                    for nodebot in nodes_bot:
                        try:
                            r = nx.multi_source_dijkstra(G,sources=[nodebot],target=nodetop)
                            print('Good path {:} to {:}'.format(nodebot,nodetop));
                            patharr = np.array(np.unravel_index(raveled_graph_idx_node[r[1]],shape=img.shape));
                            ax.plot( patharr[1,:]+0, patharr[0,:]+0 , color='red' )
                            ntracebacks+=1;
                        except nx.NetworkXNoPath as e:
                            print('No path {:} to {:}'.format(nodebot,nodetop));
                ax.text(
                    x=0.05,  # x-coordinate (0 is left, 1 is right)
                    y=0.5,  # y-coordinate (0 is bottom, 1 is top)
                    s="Num Paths\nacross {:d}".format(ntracebacks),  # Text string
                    fontsize=15,
                    ha="left",  # Horizontal alignment
                    va="top",  # Vertical alignment
                    color='yellow',
                    transform=ax.transAxes # Specify that coordinates are axis-normalized
                )


            ax = fig.add_subplot(gs[2,1]);
            ax.set_title("#3. MCP Result Find Costs",color='magenta',y=0.95,verticalalignment='top')
            ax.imshow(ret[0],aspect='auto',cmap='jet')


            ax = fig.add_subplot(gs[3,1]);
            ax.imshow(ret[1],aspect='auto',cmap='jet')
            ax.set_title("#4. MCP Result Traceback (shortest path)",color='magenta',y=0.95,verticalalignment='top')
            goodtraceback = True;

    if(not goodtraceback):
        ax = fig.add_subplot(gs[2,1]);
        ax.set_title("#3. MCP Result Find Costs",color='magenta',y=0.95,verticalalignment='top')
        ax = fig.add_subplot(gs[3,1]);
        ax.set_title("#4. MCP Result Traceback (shortest path)",color='magenta',y=0.95,verticalalignment='top')

    # plot the original unmodified image (for comparison)
    if True:
        #octstudy.pv
        #sitk.Project
        roi3_slice_y = slice(sliceinfo['pixel_depth_strip_top'].item(),sliceinfo['pixel_depth_strip_top'].item()+sliceinfo['final_wax_bot_subset_px'].item());
        #roi3_slice_y = slice(sliceinfo['wax_top_seed_candidate_px'][0],sliceinfo['wax_top_seed_candidate_px'][0]+sliceinfo['final_wax_bot_subset_px'].item());
        roi3_slice_x = slice(sliceinfo['px_wax_transverse_edges'][0],sliceinfo['px_wax_transverse_edges'][1]);
        print(roi3_slice_x)
        print(roi3_slice_y)

        #rect2 = matplotlib.patches.Rectangle( (slicey.start, slicex.start), (slicey.start+dfstepB.loc[slice_position_along_strip_px]['final_wax_bot_subset_px'] )-slicey.start, slicex.stop-slicex.start, linewidth=1, edgecolor='yellow', linestyle='dotted',facecolor='none')

        slab_thickness = 10; # px, this was used in the bulk analysis
        slab_slicer = slice(slice_position_along_strip_px-slab_thickness//2,slice_position_along_strip_px+slab_thickness//2);
        simg_slab_img = sitk.MeanProjection((simgmerged[:,slicer_1,:])[:,slab_slicer,:],projectionDimension=1);
        ndaslab = np.squeeze(sitk.GetArrayViewFromImage(simg_slab_img));

        ax = fig.add_subplot(gs[0,0]);
        ax.imshow(ndaslab,cmap='gray',aspect='auto')
        ax.set_title('#0a. Original Slab w/ Wax ROI3',color='magenta',y=0.95,verticalalignment='top')
        rect = matplotlib.patches.Rectangle( (roi3_slice_y.start, roi3_slice_x.start), roi3_slice_y.stop-roi3_slice_y.start, roi3_slice_x.stop-roi3_slice_x.start, linewidth=1, edgecolor='yellow', facecolor='none',linestyle='dashed')
        ax.add_patch(rect)

        ax = fig.add_subplot(gs[1,0]);
        ax.set_title('#0b. Original Slab w/ Overlayed Segmentation',color='magenta',y=0.95,verticalalignment='top')
        ax.imshow(ndaslab,cmap='gray',aspect='auto')
        ax.imshow(sliceseg,alpha=(sliceseg).astype(float)*0.8,cmap='jet',aspect='auto',extent=(roi3_slice_y.start,roi3_slice_y.stop,roi3_slice_x.stop,roi3_slice_x.start),origin='upper')
        ax.set_xlim(0, ndaslab.shape[1])
        ax.set_ylim(ndaslab.shape[0],0)

        ax = fig.add_subplot(gs[2,0]);
        ax.set_title('#0c. Original Slab w/ Overlayed Cost',color='magenta',y=0.95,verticalalignment='top')
        ax.imshow(ndaslab,cmap='gray',aspect='auto')
        ax.imshow(img,alpha=(img).astype(float)*0.8,cmap='jet',aspect='auto',extent=(roi3_slice_y.start,roi3_slice_y.stop,roi3_slice_x.stop,roi3_slice_x.start),origin='upper')
        ax.set_xlim(0, ndaslab.shape[1])
        ax.set_ylim(ndaslab.shape[0],0)
        #simg_slab_img = [        img_subset = simgmerged_longcropped[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]]
        #img_subset = simgmerged_longcropped[:,slice_position_along_strip_px:slice_position_along_strip_px+1,:]
        pass;
    
    # gather some info
    data['num_paths'] = ntracebacks;

    return (data,fig);
#mkfig(slice_position_along_strip_px=2065)
#mkfig(slicepx=313)
#mkfig(slice_position_along_strip_px)



In [9]:
# Run over and over
# loop and calculate and make figures

mkplot=True;
overwrite=True;

if overwrite:
    datalist = [];
#np.linspace(start=slab_thickness//2,stop=Seg.GetSize()[1],num)
#slice_centers = list( range( slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness//2 ) )
#slice_centers = list(range(slab_thickness//2, seg_sliced.GetSize()[1], slab_thickness))
import os
#filebase = r'figureoutput_{:}_stepA'.format(time.strftime('%Y%m%d'));
filebase = r'figout_post_maze_test'.format(time.strftime('%Y%m%d'));
#for count,sliceidx in enumerate(range(dfsteps.shape[0])):
for count,sliceidx in enumerate(dfsteps.index):
    print(count,sliceidx)
    output_filename = folder_figure_temp/'{:s}{:04d}.jpg'.format(filebase,count);

    if( (overwrite and os.path.exists(output_filename)) or (not os.path.exists(output_filename))):
        data,fig = mkfig(sliceidx);
        
        if mkplot:
            fig.savefig(output_filename);
            print('Wrote',output_filename);
            if(data['num_paths']>0):
                fig.savefig((octstudy.folder_study_processed/r'{:s}_connection_at_{:d}px.jpg'.format(filebase,sliceidx)));
            plt.close(fig);
        
        datalist.append(data);
        
        if(count%500 == 0):
            # try garbage collecting in the middle of the process
            # mitigate the memory leak I saw??
            import gc
            gc.collect()

0 5
Shape (28, 216)
slice(91.55555555555556, 120.11111111111111, None)
slice(208, 424, None)
Wrote C:\TEMP\OCTtmp\GHL_pyapp_20250505T1531\figout_post_maze_test0000.jpg
1 10
Shape (55, 216)
Min Top Col inf
Max Top Col inf
idxmin_end_col path 0
idxmax_end_col path 0
slice(62.327586206896555, 117.39795918367346, None)
slice(203, 419, None)
Wrote C:\TEMP\OCTtmp\GHL_pyapp_20250505T1531\figout_post_maze_test0001.jpg
2 15
Shape (51, 216)
Min Top Col inf
Max Top Col inf
idxmin_end_col path 0
idxmax_end_col path 0
slice(64.92857142857143, 116.4140625, None)
slice(204, 420, None)
Wrote C:\TEMP\OCTtmp\GHL_pyapp_20250505T1531\figout_post_maze_test0002.jpg
3 20
Shape (51, 217)
Min Top Col inf
Max Top Col inf
idxmin_end_col path 0
idxmax_end_col path 0
slice(65.9364406779661, 116.83653846153847, None)
slice(203, 420, None)
Wrote C:\TEMP\OCTtmp\GHL_pyapp_20250505T1531\figout_post_maze_test0003.jpg
4 25
Shape (55, 220)
Min Top Col inf
Max Top Col inf
idxmin_end_col path 0
idxmax_end_col path 0
slice(6

In [10]:
# make video from image sequence
import ffmpeg
import glob
import os
try:
    (
    ffmpeg
    #.input(r'C:\TEMP\{:s}%04d.jpg'.format(filebase), framerate=30) # Assumes images are named frame_1.png, frame_2.png etc.
    .input((folder_figure_temp/f'{filebase}%04d.jpg').as_posix())
    #.output(r'C:\TEMP\{:s}.mp4'.format(filebase), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .output((octstudy.folder_study_processed/r'{:s}.mp4'.format(filebase)).as_posix(), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    #.output((folder_figure_temp/r'{:s}.mp4'.format(filebase)).as_posix(), crf=20, preset='slower', movflags='faststart', pix_fmt='yuv420p')
    .run(capture_stdout=True, capture_stderr=True, overwrite_output=True)
    )
except ffmpeg.Error as e:
    print('stdout:', e.stdout.decode('utf8'))
    print('stderr:', e.stderr.decode('utf8'))
    raise e
# delete the source .jpgs
#for filepath in glob.glob(r'C:\TEMP\{:s}*.jpg'.format(filebase)):
for filepath in (folder_figure_temp).glob(f'{filebase}*.jpg'):
    os.unlink(filepath);


In [11]:
# add to dataframe
dftmp = pd.DataFrame(datalist,index=dfsteps.index)

In [12]:
# write this table to the hdf5 file
#dftmp.to_hdf(octstudy.folder_study_processed/'along_strip_data_extracted2.hdf5',key='dfstepE')
dftmp.to_hdf(octstudy.folder_study_processed/'along_strip_data_extracted.hdf5',key='dfstepE')